In [1]:
"""
The purpose of this Jupyter notebook is to perform the
training-validation-test split for positive-unlabeled (PU) learning.

Bear in mind that prior to performing the split, genes invovled in the
PPI training set must be removed.
"""

'\nThe purpose of this Jupyter notebook is to perform the\ntraining-validation-test split for positive-unlabeled (PU) learning.\n\nBear in mind that prior to performing the split, genes invovled in the\nPPI training set must be removed.\n'

In [5]:
import os
import ast
from collections import defaultdict

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.spatial.distance import jensenshannon

# Loading Data

In [6]:
# Load the screen TSV file into a Pandas DataFrame
screen_path = (
    "/Users/jacobanter/Documents/Code/VACV_screen/Dharmacon_pooled_"
    "genome_1_and_2_subset_with_control-based_Z_scoring/Dharmacon_"
    "pooled_G1_G2_screening_plates_subset_with_missing_UniProt_IDs_"
    "control-based_Z-scored.tsv"
)

screen_df = pd.read_csv(
    screen_path,
    sep="\t"
)

In [7]:
# Load the training set
training_set_path = (
    "/Users/jacobanter/Documents/Code/VACV_screen/HVIDB_pos_instances_"
    "with_nucleolus_neg_instances/new_combined_data_set_creation/data_"
    "set_files/bullet-proof_training_set.tsv"
)

training_set_df = pd.read_csv(
    training_set_path,
    sep="\t"
)

In [8]:
# Load the TSV file mapping confirmed/reliable VACV host factors to
# their categories
hf_with_category_path = "confirmed_VACV_host_factors_with_categories.tsv"

hf_with_category_df = pd.read_csv(
    hf_with_category_path,
    sep="\t"
)

# Extracting Features from the Screen DataFrame

In [9]:
# Filter the screen DataFrame to retain only rows associated with
# proteins
# Also exclude controls by selecting the well type `POOLED_SIRNA`
screen_df = screen_df.loc[
    (screen_df["WellType"] == "POOLED_SIRNA")
    &
    (screen_df["UniProt_IDs"].notna())
]

assert screen_df["Name"].notna().all(), (
    "There indeed are empty cells in the `Name` column!"
)

In [10]:
# Extract the columns of interest from the screen DataFrame
# These are the gene name (`Name`), the UniProt accessions
# (`UniProt_IDs`), the early intensity
# (`dIntensity_cPathogen_eMean_oVoronoiCells_nZScore`), the late
# intensity (`dIntensity_cLatePathogen_eMean_oVoronoiCells_nZScore`) and
# the cell count (`eCount_oCells_nZScore`)
screen_df = screen_df[[
    "Name",
    "UniProt_IDs",
    "dIntensity_cPathogen_eMean_oVoronoiCells_nZScore",
    "dIntensity_cLatePathogen_eMean_oVoronoiCells_nZScore",
    "eCount_oCells_nZScore"
]]

In [11]:
# The screen TSV file contains replicates for each gene
# Thus, for each interrogated gene, the mean feature is computed
# To this end, the DataFrame is grouped by the gene name (and also by
# the UniProt accessions)
mean_features_df = (
    screen_df
    .groupby(["Name", "UniProt_IDs"], as_index=False)
    .mean()
)

In [12]:
# Also explode the `UniProt_IDs` column
mean_features_df["UniProt_IDs"] = (
    mean_features_df["UniProt_IDs"].str.split(";")
)

mean_features_df = mean_features_df.explode("UniProt_IDs")

In [13]:
# Save the DataFrame with the mean features to disk
mean_features_df.to_csv(
    "screen_subset_mean_features.tsv",
    sep="\t",
    index=False
)

# Removing Genes involved in the PPI Training Set

In [14]:
# Removing genes involved in the PPI trainig set with the intention to
# prevent label/target leakage is a bit overcautious and not necessary

In [15]:
mean_features_df = pd.read_csv(
    "screen_subset_mean_features.tsv",
    sep="\t"
)

In [ ]:
# print(
#     f"DataFrame length prior to filtering: {len(mean_features_df):,}"
# )

DataFrame length prior to filtering: 44,269


In [ ]:
# # Determine the unique human UniProt accessions in the PPI training set
# human_uniprot_accs_in_ppi_training = training_set_df["Human_prot"].unique()

In [ ]:
# # Remove the human UniProt accessions present in the PPI training set
# # from `mean_features_df`
# mean_features_df = mean_features_df.loc[
#     ~mean_features_df["UniProt_IDs"]
#     .isin(human_uniprot_accs_in_ppi_training)
# ]

In [ ]:
# print(
#     f"DataFrame length after filtering: {len(mean_features_df):,}"
# )

DataFrame length after filtering: 43,700


# Constructing the DataFrame for the Train-Validation-Test Split

In [16]:
# A sophisticated procedure has been devised for the
# train-validation-test split
# This procedure requires a specific input format
# In the following, a DataFrame meeting this specific input format is
# constructed
df_for_split = mean_features_df.copy()

# Specifically, the input format requires the phenotype features to be
# stored in one list as phenotype vector (i.e. for each row/protein, the
# early/late intensity and cell count are stored in one list)
df_for_split["phenotype_vec"] = df_for_split.apply(
    lambda row: [
        row["dIntensity_cPathogen_eMean_oVoronoiCells_nZScore"],
        row["dIntensity_cLatePathogen_eMean_oVoronoiCells_nZScore"],
        row["eCount_oCells_nZScore"]
    ],
    axis=1
)

df_for_split.drop(
    [
        "dIntensity_cPathogen_eMean_oVoronoiCells_nZScore",
        "dIntensity_cLatePathogen_eMean_oVoronoiCells_nZScore",
        "eCount_oCells_nZScore"
    ],
    axis=1,
    inplace=True
)

In [17]:
# Also rename the first two columns
df_for_split.rename(
    columns={
        "Name": "gene",
        "UniProt_IDs": "protein_id"
    },
    inplace=True
)

In [18]:
# Now, add a `label` column to the DataFrame
# The `label` column indicates whether the respective gene/protein is a
# known VACV host factor or not
# Accordingly, it can assume two values (1 for known host factors, i.e.
# positive, and 0 for unlabeled instances)
vacv_host_factors = hf_with_category_df["host_factor"].unique()

df_for_split["label"] = (
    df_for_split["gene"].isin(vacv_host_factors).astype(int)
)

In [19]:
# Lastly, introduce the `hf_class` column
# The `hf_class` column, as its name already implies, indicates for
# known VACV host factors, i.e. rows with `label == 1`, the host factor
# class
# This column can assume three values, which are "proviral", "antiviral"
# and None
df_for_split = pd.merge(
    df_for_split,
    hf_with_category_df,
    left_on="gene",
    right_on="host_factor",
    how="left"
)

# Drop the `host_factor` column
df_for_split.drop(
    "host_factor",
    axis=1,
    inplace=True
)

In [20]:
# Rename the `category` column into `hf_class`
df_for_split.rename(
    columns={"category": "hf_class"},
    inplace=True
)

In [21]:
# Save the DataFrame for the split to disk
df_for_split.to_csv(
    "DataFrame_for_train_validation_test_split.tsv",
    sep="\t",
    index=False
)

# Training-Validation-Test Split

In [22]:
# Load the DataFrame to subject to the train-validation-test split
df_for_split = pd.read_csv(
    "DataFrame_for_train_validation_test_split.tsv",
    sep="\t",
    converters={"phenotype_vec": ast.literal_eval}
)

In [23]:
# Define the procedure for the training-validation-test split

# -----------------------------
# 0. Phenotype clustering
# -----------------------------
def compute_pheno_clusters(df, n_clusters=10, seed=42):
    X = np.vstack(df["phenotype_vec"].values)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    clusters = kmeans.fit_predict(X_scaled)

    df = df.copy()
    df["pheno_cluster"] = clusters
    return df


# ----------------------------------------
# 1. Collapsing the rows to the gene level
# ----------------------------------------
def collapse_to_gene_level(df):
    rows = []

    for gene, subdf in df.groupby("gene"):
        row = {}

        row["gene"] = gene
        row["label"] = int(subdf["label"].max())

        # HF class
        hf_classes = subdf["hf_class"].dropna().unique()
        if row["label"] == 1:
            row["hf_class"] = hf_classes[0] if len(hf_classes) == 1 else "mixed"
        else:
            row["hf_class"] = "unlabeled"

        # Proteins
        row["protein_ids"] = subdf["protein_id"].tolist()

        # Phenotype vector (unchanged)
        row["phenotype_vec"] = subdf["phenotype_vec"].iloc[0]

        # ✅ FIX: propagate phenotype cluster
        row["pheno_cluster"] = subdf["pheno_cluster"].mode()[0]

        rows.append(row)

    return pd.DataFrame(rows)


# -----------------------------
# 2. Build connected components
# -----------------------------
def build_gene_protein_components(gene_df):
    gene_to_proteins = {}
    protein_to_genes = defaultdict(set)

    for _, row in gene_df.iterrows():
        g = row["gene"]
        prots = row["protein_ids"]

        gene_to_proteins[g] = set(prots)
        for p in prots:
            protein_to_genes[p].add(g)

    visited = set()
    components = []

    def bfs(start_gene):
        stack = [start_gene]
        genes = set()
        proteins = set()

        while stack:
            node = stack.pop()

            if node in visited:
                continue
            visited.add(node)

            if node in gene_to_proteins:
                genes.add(node)
                for p in gene_to_proteins[node]:
                    if p not in visited:
                        stack.append(p)
            else:
                proteins.add(node)
                for g in protein_to_genes[node]:
                    if g not in visited:
                        stack.append(g)

        return genes, proteins

    for gene in gene_to_proteins:
        if gene not in visited:
            genes, proteins = bfs(gene)
            components.append({
                "genes": list(genes),
                "proteins": list(proteins)
            })

    return components


# -----------------------------
# 3. Build component dataframe
# -----------------------------
def build_component_df(components, gene_df):
    rows = []

    for i, comp in enumerate(components):
        subdf = gene_df[gene_df["gene"].isin(comp["genes"])]

        label = int(subdf["label"].max())

        if label == 1:
            hf_classes = subdf["hf_class"].unique()
            hf_class = hf_classes[0] if len(hf_classes) == 1 else "mixed"
        else:
            hf_class = "unlabeled"

        pheno_cluster = subdf["pheno_cluster"].mode()[0]

        rows.append({
            "component_id": i,
            "genes": comp["genes"],
            "label": label,
            "hf_class": hf_class,
            "pheno_cluster": pheno_cluster,
            "size": len(comp["genes"])
        })

    return pd.DataFrame(rows)


# -----------------------------
# 4. Stratified split
# -----------------------------
def split_components_balanced(comp_df, seed=42):
    np.random.seed(seed)

    comp_df = comp_df.sample(frac=1, random_state=seed)
    comp_df = comp_df.sort_values("size", ascending=False).reset_index(drop=True)

    splits = {
        "train": {"rows": [], "size": 0},
        "val": {"rows": [], "size": 0},
        "test": {"rows": [], "size": 0},
    }

    target_ratios = {"train": 0.7, "val": 0.1, "test": 0.2}
    total_size = comp_df["size"].sum()
    target_sizes = {k: v * total_size for k, v in target_ratios.items()}

    # Seed splits
    split_names = list(splits.keys())
    for i, (_, comp) in enumerate(comp_df.iloc[:3].iterrows()):
        splits[split_names[i]]["rows"].append(comp)
        splits[split_names[i]]["size"] += comp["size"]

    remaining = comp_df.iloc[3:]

    for _, comp in remaining.iterrows():
        deficits = {
            s: target_sizes[s] - splits[s]["size"]
            for s in splits
        }
        best_split = max(deficits, key=deficits.get)

        splits[best_split]["rows"].append(comp)
        splits[best_split]["size"] += comp["size"]

    return (
        pd.DataFrame(splits["train"]["rows"]),
        pd.DataFrame(splits["val"]["rows"]),
        pd.DataFrame(splits["test"]["rows"]),
    )


# -----------------------------
# 5. Expand to gene-level
# -----------------------------
def expand_components(comp_split, gene_df):
    genes = []
    for _, row in comp_split.iterrows():
        genes.extend(row["genes"])
    return gene_df[gene_df["gene"].isin(genes)].copy()


# -----------------------------
# 6. Enforce P/U ratio
# -----------------------------
def enforce_pu_ratio(df, target_ratio=5, seed=42):
    np.random.seed(seed)

    pos = df[df["label"] == 1]
    unl = df[df["label"] == 0]

    n_pos = len(pos)
    n_unl_target = int(n_pos * target_ratio)

    if n_unl_target >= len(unl):
        return df

    sampled_unl = []

    for cluster, subdf in unl.groupby("pheno_cluster"):
        frac = len(subdf) / len(unl)
        n_cluster = int(frac * n_unl_target)

        sampled_unl.append(
            subdf.sample(n=min(n_cluster, len(subdf)), random_state=seed)
        )

    sampled_unl = pd.concat(sampled_unl)

    return pd.concat([pos, sampled_unl])


# -----------------------------
# 7. Logging
# -----------------------------
def log_class_balance(df, name):
    print(f"\n[{name}]")
    print("Total:", len(df))
    print("Positives:", df["label"].sum())
    print("Unlabeled:", (df["label"] == 0).sum())
    print("HF class distribution:")
    print(df["hf_class"].value_counts(dropna=False))


def phenotype_distribution(df):
    return df["pheno_cluster"].value_counts(normalize=True).sort_index()


def compare_distributions(train_df, val_df, test_df):
    print("\n[Phenotype Distribution Similarity]")

    train_dist = phenotype_distribution(train_df)
    val_dist = phenotype_distribution(val_df)
    test_dist = phenotype_distribution(test_df)

    # Align indices
    all_idx = sorted(set(train_dist.index) | set(val_dist.index) | set(test_dist.index))
    train_dist = train_dist.reindex(all_idx, fill_value=0)
    val_dist = val_dist.reindex(all_idx, fill_value=0)
    test_dist = test_dist.reindex(all_idx, fill_value=0)

    print("JS(train || val): ", jensenshannon(train_dist, val_dist))
    print("JS(train || test):", jensenshannon(train_dist, test_dist))


# -----------------------------
# 9. Full pipeline
# -----------------------------
def create_splits(
    df,
    n_pheno_clusters=10,
    pu_ratio=5,
    seed=42
):
    print("Step 1: Phenotype clustering...")
    df = compute_pheno_clusters(df, n_pheno_clusters, seed)

    print("Step 2: Collapse to gene level...")
    gene_df = collapse_to_gene_level(df)

    print("Step 3: Building components...")
    components = build_gene_protein_components(gene_df)
    comp_df = build_component_df(components, gene_df)
    print(comp_df["size"].describe())

    print("Step 4: Balanced component splitting...")
    train_c, val_c, test_c = split_components_balanced(comp_df, seed)

    train_df = expand_components(train_c, gene_df)
    val_df = expand_components(val_c, gene_df)
    test_df = expand_components(test_c, gene_df)

    print("Step 5: Enforcing P/U ratio...")
    train_df = enforce_pu_ratio(train_df, pu_ratio, seed)
    val_df = enforce_pu_ratio(val_df, pu_ratio, seed)
    test_df = enforce_pu_ratio(test_df, pu_ratio, seed)

    assert train_df["gene"].nunique() == len(train_df)
    assert val_df["gene"].nunique() == len(val_df)
    assert test_df["gene"].nunique() == len(test_df)

    assert set(train_df["gene"]).isdisjoint(val_df["gene"])
    assert set(train_df["gene"]).isdisjoint(test_df["gene"])
    assert set(val_df["gene"]).isdisjoint(test_df["gene"])

    # Logging
    log_class_balance(train_df, "TRAIN")
    log_class_balance(val_df, "VAL")
    log_class_balance(test_df, "TEST")

    compare_distributions(train_df, val_df, test_df)

    return train_df, val_df, test_df

In [24]:
train_df, val_df, test_df = create_splits(
    df_for_split,
    pu_ratio=10
)

Step 1: Phenotype clustering...
Step 2: Collapse to gene level...
Step 3: Building components...
count    17367.000000
mean         1.017562
std          0.225574
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         12.000000
Name: size, dtype: float64
Step 4: Balanced component splitting...
Step 5: Enforcing P/U ratio...

[TRAIN]
Total: 1843
Positives: 168
Unlabeled: 1675
HF class distribution:
hf_class
unlabeled    1675
proviral      114
mixed          46
antiviral       8
Name: count, dtype: int64

[VAL]
Total: 269
Positives: 25
Unlabeled: 244
HF class distribution:
hf_class
unlabeled    244
proviral      16
mixed          7
antiviral      2
Name: count, dtype: int64

[TEST]
Total: 502
Positives: 46
Unlabeled: 456
HF class distribution:
hf_class
unlabeled    456
proviral      30
mixed         15
antiviral      1
Name: count, dtype: int64

[Phenotype Distribution Similarity]
JS(train || val):  0.04107027158778312
JS(train || test): 0.024

In [25]:
# Save the splits to disk
split_dir_1_10 = "PU_data_set_splits_PU_ratio_1_10"

if not os.path.exists(split_dir_1_10):
    os.makedirs(split_dir_1_10)

train_df.to_csv(
    os.path.join(
        split_dir_1_10,
        "PU_training_set.tsv"
    ),
    sep="\t",
    index=False
)

val_df.to_csv(
    os.path.join(
        split_dir_1_10,
        "PU_validation_set.tsv"
    ),
    sep="\t",
    index=False
)

test_df.to_csv(
    os.path.join(
        split_dir_1_10,
        "PU_test_set.tsv"
    ),
    sep="\t",
    index=False
)

In [26]:
# Repeat the training-validation-set split for different P:U ratios
# P:U ratio of 1:5
train_df, val_df, test_df = create_splits(
    df_for_split,
    pu_ratio=5
)

# Save the splits to disk
split_dir_1_5 = "PU_data_set_splits_PU_ratio_1_5"

if not os.path.exists(split_dir_1_5):
    os.makedirs(split_dir_1_5)

train_df.to_csv(
    os.path.join(
        split_dir_1_5,
        "PU_training_set.tsv"
    ),
    sep="\t",
    index=False
)

val_df.to_csv(
    os.path.join(
        split_dir_1_5,
        "PU_validation_set.tsv"
    ),
    sep="\t",
    index=False
)

test_df.to_csv(
    os.path.join(
        split_dir_1_5,
        "PU_test_set.tsv"
    ),
    sep="\t",
    index=False
)

Step 1: Phenotype clustering...
Step 2: Collapse to gene level...
Step 3: Building components...
count    17367.000000
mean         1.017562
std          0.225574
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         12.000000
Name: size, dtype: float64
Step 4: Balanced component splitting...
Step 5: Enforcing P/U ratio...

[TRAIN]
Total: 1003
Positives: 168
Unlabeled: 835
HF class distribution:
hf_class
unlabeled    835
proviral     114
mixed         46
antiviral      8
Name: count, dtype: int64

[VAL]
Total: 145
Positives: 25
Unlabeled: 120
HF class distribution:
hf_class
unlabeled    120
proviral      16
mixed          7
antiviral      2
Name: count, dtype: int64

[TEST]
Total: 272
Positives: 46
Unlabeled: 226
HF class distribution:
hf_class
unlabeled    226
proviral      30
mixed         15
antiviral      1
Name: count, dtype: int64

[Phenotype Distribution Similarity]
JS(train || val):  0.05363749827942746
JS(train || test): 0.03092320

In [27]:
# P:U ratio of 1:20
train_df, val_df, test_df = create_splits(
    df_for_split,
    pu_ratio=20
)

# Save the splits to disk
split_dir_1_20 = "PU_data_set_splits_PU_ratio_1_20"

if not os.path.exists(split_dir_1_20):
    os.makedirs(split_dir_1_20)

train_df.to_csv(
    os.path.join(
        split_dir_1_20,
        "PU_training_set.tsv"
    ),
    sep="\t",
    index=False
)

val_df.to_csv(
    os.path.join(
        split_dir_1_20,
        "PU_validation_set.tsv"
    ),
    sep="\t",
    index=False
)

test_df.to_csv(
    os.path.join(
        split_dir_1_20,
        "PU_test_set.tsv"
    ),
    sep="\t",
    index=False
)

Step 1: Phenotype clustering...
Step 2: Collapse to gene level...
Step 3: Building components...
count    17367.000000
mean         1.017562
std          0.225574
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         12.000000
Name: size, dtype: float64
Step 4: Balanced component splitting...
Step 5: Enforcing P/U ratio...

[TRAIN]
Total: 3523
Positives: 168
Unlabeled: 3355
HF class distribution:
hf_class
unlabeled    3355
proviral      114
mixed          46
antiviral       8
Name: count, dtype: int64

[VAL]
Total: 519
Positives: 25
Unlabeled: 494
HF class distribution:
hf_class
unlabeled    494
proviral      16
mixed          7
antiviral      2
Name: count, dtype: int64

[TEST]
Total: 961
Positives: 46
Unlabeled: 915
HF class distribution:
hf_class
unlabeled    915
proviral      30
mixed         15
antiviral      1
Name: count, dtype: int64

[Phenotype Distribution Similarity]
JS(train || val):  0.033441169246649874
JS(train || test): 0.02